In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

df = pd.read_csv(ROOT / "data" / "processed" / "features_with_causal_states.csv")
STATES = ["Stable", "Strained", "Critical"]
rank = {s: i for i, s in enumerate(STATES)}
train, test = df[df.split == "train"], df[df.split == "test"]

def report(name, pred):
    d = np.abs(np.array([rank[p] for p in pred]) - np.array([rank[s] for s in test.true_state]))
    return {"method": name, "exact": round((d == 0).mean(), 3),
            "within_one": round((d <= 1).mean(), 3),
            "stable_vs_critical_mixups": round((d == 2).mean(), 3)}

rows = [report("always guess the most common", [train.true_state.mode()[0]] * len(test)),
        report("HMM (current)", test.hmm_state_causal)]

# Simple rule: thresholds on the 3-month income/spending ratio, learned on training households only.
x_tr, x_te = train.ratio_roll3.values, test.ratio_roll3.values
cut = lambda x, a, b: np.where(x < a, "Critical", np.where(x < b, "Strained", "Stable"))
qs = np.quantile(x_tr, np.linspace(0.05, 0.95, 37))
a, b = max(((a, b) for a in qs for b in qs if a < b),
           key=lambda ab: (cut(x_tr, *ab) == train.true_state.values).mean())
rows.append(report("rule on 3-month income/spending ratio", cut(x_te, a, b)))

# Random Forest classifier: learns from training households where the true state is known.
feat = [c for c in df.select_dtypes("number").columns if c != "month"]
clf = RandomForestClassifier(n_estimators=300, min_samples_leaf=5, random_state=0)
clf.fit(train[feat], train.true_state)
rows.append(report("Random Forest classifier", clf.predict(test[feat])))

pd.DataFrame(rows)

,method,exact,within_one,stable_vs_critical_mixups
0,always guess the most common,0.378,1.000,0.000
1,HMM (current),0.531,0.936,0.064
2,rule on 3-month income/spending ratio,0.532,0.896,0.104
3,Random Forest classifier,0.799,0.993,0.007


In [2]:
imp = pd.Series(clf.feature_importances_, index=feat).sort_values(ascending=False)
imp.head(10).round(3)

income_to_expenditure_ratio    0.162
savings_share                  0.106
entertainment_share            0.090
ratio_roll3                    0.077
overspend_flag                 0.068
savings_rate                   0.058
savings                        0.057
overspend_roll3                0.051
entertainment                  0.041
transport_share                0.022
dtype: float64

In [3]:
import joblib
joblib.dump({"model": clf, "features": feat, "states": STATES},
            ROOT / "models" / "state_classifier.joblib")
print("saved:", (ROOT / "models" / "state_classifier.joblib").stat().st_size // 1024, "KB")


saved: 9727 KB
